# Phase 8 — Insight Discovery

This notebook is the analytical layer — asking real questions about the placement data and finding real answers. It is the evidence that the data was analysed as a journalist, not just processed as an engineer.

**Source:** `data/processed/HirePrism.duckdb`  
**Output:** Findings feed into `data/insights/insight_report.json` and `insight_report.md`

In [ ]:
import sys
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import numpy as np

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DB = ROOT / 'data' / 'processed' / 'HirePrism.duckdb'
con = duckdb.connect(str(DB), read_only=True)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print('Connected to', DB.name)

## Q1: Does CTC change across the placement season?

In [ ]:
trend = con.execute("""
    SELECT STRFTIME(notice_date,'%Y-%m') AS month,
           COUNT(*) AS offers,
           ROUND(AVG(ctc_lpa_normalized), 2) AS avg_ctc
    FROM fact_offers
    WHERE ctc_status IN ('KNOWN','RANGE') AND notice_date IS NOT NULL
    GROUP BY 1 ORDER BY 1
""").df()

fig, ax1 = plt.subplots(figsize=(11, 4))
ax2 = ax1.twinx()
ax1.plot(trend['month'], trend['avg_ctc'], 'o-', color='steelblue', linewidth=2, label='Avg CTC (LPA)')
ax2.bar(trend['month'], trend['offers'], alpha=0.25, color='gray', label='Offer count')
ax1.set_ylabel('Average CTC (LPA)', color='steelblue')
ax2.set_ylabel('Offer count', color='gray')
ax1.set_title('CTC Trend Across Placement Season')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"Peak: {trend.loc[trend['avg_ctc'].idxmax(), 'month']} at {trend['avg_ctc'].max()} LPA")
print(f"Trough: {trend.loc[trend['avg_ctc'].idxmin(), 'month']} at {trend['avg_ctc'].min()} LPA")
print(f"Drop: {round((trend['avg_ctc'].max()-trend['avg_ctc'].min())/trend['avg_ctc'].max()*100,1)}%")

**Finding:** July–August 2025 offers averaged 18–26 LPA. By Q1 2026, the average had fallen to 6–8 LPA — a ~74% decline. Early-season placements are dominated by product/tech companies; late-season by manufacturing and core engineering.

## Q2: Which offer type pays the most? PPO vs FTE vs Intern

In [ ]:
otype = con.execute("""
    SELECT offer_type_standardized, COUNT(*) AS n,
           ROUND(AVG(ctc_lpa_normalized),2) AS avg_ctc,
           ROUND(MIN(ctc_lpa_normalized),2) AS min_ctc,
           ROUND(MAX(ctc_lpa_normalized),2) AS max_ctc
    FROM fact_offers WHERE ctc_status IN ('KNOWN','RANGE')
    GROUP BY offer_type_standardized ORDER BY avg_ctc DESC
""").df()

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(otype['offer_type_standardized'], otype['avg_ctc'],
               color=['#d73027','#f46d43','#fdae61','#74add1','#4575b4'])
ax.set_xlabel('Average CTC (LPA)')
ax.set_title('Average CTC by Offer Type (Known CTC Only)')
for bar, val in zip(bars, otype['avg_ctc']):
    ax.text(bar.get_width()+0.2, bar.get_y()+bar.get_height()/2,
            f'{val} LPA', va='center')
plt.tight_layout(); plt.show()
print(otype.to_string(index=False))

**Finding:** PPOs average 22.93 LPA — 2.38× the FTE average of 9.63 LPA. The intern-FTE pathway also outperforms direct FTE by 32%. If a student can secure a PPO or intern-FTE offer, the financial upside is substantial.

## Q3: Which branches have the best FTE-to-intern ratio?

In [ ]:
branch_ratio = con.execute("""
    SELECT branch_group,
           SUM(fte_count) AS fte, SUM(intern_count) AS intern,
           ROUND(SUM(fte_count)*1.0/NULLIF(SUM(intern_count),0),2) AS ratio,
           SUM(offer_count) AS total
    FROM vw_branch_summary
    WHERE branch_group NOT IN ('ALL','NA','UNKNOWN')
    GROUP BY branch_group ORDER BY ratio DESC
""").df()

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#2ecc71' if r >= 0.5 else '#e74c3c' for r in branch_ratio['ratio']]
ax.barh(branch_ratio['branch_group'], branch_ratio['ratio'], color=colors)
ax.axvline(1.0, color='black', linestyle='--', linewidth=1, label='Equal FTE/Intern')
ax.set_xlabel('FTE-to-Intern Ratio')
ax.set_title('FTE-to-Intern Ratio by Branch Group')
ax.legend()
plt.tight_layout(); plt.show()
print(branch_ratio.to_string(index=False))

**Finding:** Only CIVIL (1.14) has more FTE offers than intern offers. CS (0.23) and BIO (0.22) are the most intern-heavy. This reflects the tech industry's trial-before-hire culture vs manufacturing/civil where FTE hiring is direct.

## Q4: Is there a correlation between how much a company recruits and how much it pays?

In [ ]:
co = con.execute("""
    SELECT total_offers, avg_ctc_lpa FROM vw_company_summary
    WHERE avg_ctc_lpa IS NOT NULL
""").df()

r = co['total_offers'].corr(co['avg_ctc_lpa'])
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(co['total_offers'], co['avg_ctc_lpa'], alpha=0.4, s=20, color='steelblue')
ax.set_xlabel('Number of Offers Posted')
ax.set_ylabel('Average CTC (LPA)')
ax.set_title(f'Company Offer Volume vs Average CTC  (Pearson r = {r:.3f})')
plt.tight_layout(); plt.show()
print(f'Correlation: {r:.3f} — effectively zero relationship')

**Finding:** r = 0.010 — no correlation. High-volume recruiters are not systematically higher or lower paying. Company quality and pay are orthogonal to recruit volume in this dataset.

## Q5: Which role families have the highest CTC variance?

In [ ]:
variance = con.execute("""
    SELECT job_family, COUNT(*) AS n,
           ROUND(AVG(ctc_lpa_normalized),2) AS avg_ctc,
           ROUND(STDDEV(ctc_lpa_normalized),2) AS std_ctc,
           ROUND(STDDEV(ctc_lpa_normalized)/NULLIF(AVG(ctc_lpa_normalized),0),3) AS cv
    FROM fact_offers WHERE ctc_status IN ('KNOWN','RANGE') AND job_family != 'Unknown'
    GROUP BY job_family HAVING COUNT(*)>=5 ORDER BY cv DESC
""").df()

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(variance['job_family'], variance['cv'], color='#8e44ad')
ax.set_xlabel('Coefficient of Variation (std/mean)')
ax.set_title('CTC Variance by Job Family (Higher = More Spread)')
plt.tight_layout(); plt.show()
print(variance[['job_family','avg_ctc','std_ctc','cv','n']].to_string(index=False))

**Finding:** Software Engineering (CV=0.91) has by far the widest pay spread — from 3.6 LPA service sector jobs to 123 LPA outlier tech roles. Research (CV=0.13) is the most stable and predictable.

## Q6: Which role families waive CGPA most often?

In [ ]:
cgpa = con.execute("""
    SELECT job_family, offer_count, no_cgpa_count,
           ROUND(no_cgpa_count*100.0/offer_count,1) AS no_cgpa_pct
    FROM vw_role_summary WHERE offer_count >= 5 ORDER BY no_cgpa_pct DESC
""").df()

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(cgpa['job_family'], cgpa['no_cgpa_pct'], color='#27ae60')
ax.set_xlabel('% of Offers with No CGPA Requirement')
ax.set_title('No-CGPA Rate by Job Family')
for i, (_, r) in enumerate(cgpa.iterrows()):
    ax.text(r['no_cgpa_pct']+0.5, i, f"{r['no_cgpa_pct']}%", va='center')
plt.tight_layout(); plt.show()

**Finding:** Data Engineering (63.6%) and Data/Analytics (34.2%) waive CGPA most often, reflecting a skills-first hiring philosophy. Core Engineering waives CGPA least often (14.3%), consistent with GPA-gated campus drives.

## Q7: Which companies are the best intern-to-FTE pathways?

In [ ]:
intern_fte = con.execute("""
    SELECT company_name, COUNT(*) AS intern_fte_offers,
           ROUND(AVG(ctc_lpa_normalized),2) AS avg_ctc
    FROM fact_offers
    WHERE offer_type_standardized='INTERN_FTE'
      AND ctc_status IN ('KNOWN','RANGE')
    GROUP BY company_name HAVING COUNT(*)>=2
    ORDER BY avg_ctc DESC LIMIT 10
""").df()
print(intern_fte.to_string(index=False))

**Finding:** Meesho (46.83 LPA avg) is the standout for intern-FTE compensation — 3.7× the sector average of 12.75 LPA. Addverb Technologies offers the most intern-FTE positions (6 offers at 16 LPA avg).

## Q8: How accessible are high-package offers to students without strong CGPA?

In [ ]:
hp_no_cgpa = con.execute("""
    SELECT COUNT(*) AS high_pkg_no_cgpa,
           (SELECT COUNT(*) FROM fact_offers WHERE ctc_lpa_normalized>=10
            AND ctc_status IN ('KNOWN','RANGE')) AS total_high_pkg
    FROM fact_offers
    WHERE no_cgpa_criteria=TRUE AND ctc_lpa_normalized>=10
      AND ctc_status IN ('KNOWN','RANGE')
""").df()

n_hp_nc = hp_no_cgpa['high_pkg_no_cgpa'].iloc[0]
n_hp = hp_no_cgpa['total_high_pkg'].iloc[0]
print(f'High-package (≥10 LPA) + No CGPA filter: {n_hp_nc} out of {n_hp} high-pkg offers ({round(n_hp_nc/n_hp*100,1)}%)')
print()
print('Top 10 accessible high-package offers:')
print(con.execute("""
    SELECT company_name, job_family, ctc_lpa_normalized, offer_type_standardized
    FROM vw_high_package_offers WHERE no_cgpa_criteria=TRUE
    ORDER BY ctc_lpa_normalized DESC LIMIT 10
""").df().to_string(index=False))

**Finding:** 47 of 247 high-package offers (19%) require no CGPA. This is a significant opportunity pool — nearly 1 in 5 premium offers is accessible regardless of academic score.

In [ ]:
con.close()
print('Notebook complete. Findings exported to data/insights/.')